# Assignment 20: Building & Deploying a Content-Based Recommendation System

In Assignment 19 I built Word2Vec embeddings to capture semantic relationships between words in a continuous vector space. Here the goal is different: instead of learning word vectors from scratch, I use TF-IDF to vectorize company descriptions, compute pairwise cosine similarity between them, and build an end-to-end recommendation engine that suggests similar companies based on business descriptions.

Dataset Name: Open Data 500 US Companies (us_companies.csv)
Kaggle Link: https://www.kaggle.com/datasets/thegovlab/open-data-500-companies
Stack: Python, Pandas, scikit-learn, Streamlit, Git/GitHub, Render.


In [1]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
nltk.download('stopwords', quiet=True)
print("Libraries loaded successfully")

Libraries loaded successfully


## PART 1 - Data Preprocessing
## Task 1: Load and Understand Dataset

- load us_companies.csv with multi-path check so it runs locally and in repo
- check shape, columns, first 5 rows, nulls, duplicates, and top categories
- pick item id column and text columns for recommendation


In [2]:
# multi-path check for dataset across working directories
possible_paths = [
    'data/us_companies.csv', 
    'content-based-recommender/data/us_companies.csv', 
    '../data/us_companies.csv', 
    'us_companies.csv'
]

dataset_path = None
for path in possible_paths:
    if os.path.exists(path):
        dataset_path = path
        break

if dataset_path is None:
    raise FileNotFoundError("Could not locate us_companies.csv in candidate paths.")

df = pd.read_csv(dataset_path)
print(f"Loaded from: {dataset_path} | Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Loaded from: data/us_companies.csv | Shape: (529, 22)
Columns: ['company_name_id', 'company_name', 'url', 'year_founded', 'city', 'state', 'country', 'zip_code', 'full_time_employees', 'company_type', 'company_category', 'revenue_source', 'business_model', 'social_impact', 'description', 'description_short', 'source_count', 'data_types', 'example_uses', 'data_impacts', 'financial_info', 'last_updated']


,company_name_id,company_name,url,year_founded,city,state,country,zip_code,full_time_employees,company_type,...,business_model,social_impact,description,description_short,source_count,data_types,example_uses,data_impacts,financial_info,last_updated
0,3-round-stones-inc,"3 Round Stones, Inc.",http://3RoundStones.com,2010.0,Washington,DC,us,20004.0,1-10,Private,...,"Business to Business, Business to Consumer",NaN,3 Round Stones produces a platform for publish...,Our Open Source platform is used by the Fortun...,NaN,NaN,NaN,[],"3 Round Stones is a profitable, self-funded, w...",2014-11-12 14:44:25.969871
1,48-factoring-inc,48 Factoring Inc.,https://www.48factoring.com,2014.0,Philadelphia,PA,us,19087.0,51-200,Private,...,Business to Business,Small Business Owners,The company mission is to provide finance to s...,48 Factoring Inc. is one of the best financial...,11-50,Business,NaN,"[u'Cost efficiency', u'Job growth', u'Revenue ...",NaN,2015-05-18 11:36:39.902880
2,5psolutions,5PSolutions,www.5psolutions.com,2007.0,Fairfax,VA,us,22003.0,1-10,Private,...,"Business to Business, Business to Consumer, Bu...",NaN,"At 5PSolutions, we wish to make all basic info...",5PSolutions are artisans of mobile platforms.,NaN,NaN,NaN,[],NaN,2014-11-12 15:09:35.485949
3,abt-associates,Abt Associates,abtassoc.com,1965.0,Cambridge,MA,us,2138.0,"1,001-5,000",Private,...,NaN,NaN,"Abt Associates is a mission-driven, internatio...","Abt Associates is a mission-driven, global lea...",101+,NaN,NaN,[],Employee-owned company. $552M/year.,2014-09-11 17:23:21.390289
4,accela,Accela,http://www.accela.com,1999.0,San Ramon,CA,us,94583.0,"501-1,000",Private,...,Business to Consumer,"Citizen engagement and participation, Good gov...",Accela powers thousands of services and millio...,Accela improves citizen engagement by making i...,101+,"Health/Healthcare, Housing, Public Safety",NaN,[],NaN,2014-09-16 16:30:49.172802


In [3]:
# Check total rows, unique company names, duplicates, missing values, and top categories.
key_cols = ['company_name', 'company_category', 'description', 'description_short']
print(f"Total companies: {len(df)} | Unique names: {df['company_name'].nunique()}")
print(f"Duplicate rows: {df.duplicated().sum()}\n")
print("Missing values:\n", df[key_cols].isnull().sum())
print("\nTop 10 categories:\n", df['company_category'].value_counts().head(10))

Total companies: 529 | Unique names: 529
Duplicate rows: 0

Missing values:
 company_name         0
company_category     3
description          0
description_short    0
dtype: int64

Top 10 categories:
 company_category
Data/Technology              97
Finance & Investment         75
Business & Legal Services    44
Governance                   43
Healthcare                   40
Geospatial/Mapping           30
Research & Consulting        28
Transportation               28
Energy                       28
Lifestyle & Consumer         25
Name: count, dtype: int64


In [4]:
# Set the item identifier and text feature columns used for recommendations.
item_id_col = 'company_name'
text_cols = ['description', 'company_category', 'description_short']
print(f"Item ID: {item_id_col} | Text features: {text_cols}")
df[[item_id_col] + text_cols].head(3)

Item ID: company_name | Text features: ['description', 'company_category', 'description_short']


,company_name,description,company_category,description_short
0,"3 Round Stones, Inc.",3 Round Stones produces a platform for publish...,Data/Technology,Our Open Source platform is used by the Fortun...
1,48 Factoring Inc.,The company mission is to provide finance to s...,Finance & Investment,48 Factoring Inc. is one of the best financial...
2,5PSolutions,"At 5PSolutions, we wish to make all basic info...",Data/Technology,5PSolutions are artisans of mobile platforms.


- 529 unique companies, 22 columns, 0 duplicate rows
- categories: Data/Technology (97), Finance & Investment (75), Business & Legal Services (44)
- nulls: 0 in description and description_short, 3 missing in company_category
- text features: description + company_category + description_short combined

### Task 2: Text Preprocessing for Recommendation

Preprocessing steps:
- fill missing values with empty string
- join description, company_category, and description_short into one string
- lowercase text, strip punctuation/numbers with regex [^a-z\s], remove NLTK English stopwords and 1-letter tokens
- save output in clean_text column


In [5]:
# Fill missing values, join the three text columns, and clean tokens into clean_text.
stop_words = set(stopwords.words('english'))
for col in ['description', 'company_category', 'description_short']:
    df[col] = df[col].fillna('')

df['combined_text'] = df['description'] + ' ' + df['company_category'] + ' ' + df['description_short']

def clean_text(text):
    text = re.sub(r'[^a-z\s]', ' ', text.lower())
    return ' '.join([w for w in text.split() if len(w) > 1 and w not in stop_words])

df['clean_text'] = df['combined_text'].apply(clean_text)

# Verify zero nulls in clean_text and compare original vs cleaned text samples.
print(f"Missing values in clean_text: {df['clean_text'].isna().sum()}")
for i in range(3):
    print(f"\n{df['company_name'].iloc[i]}")
    print(f"Raw  : {df['combined_text'].iloc[i][:100]}...")
    print(f"Clean: {df['clean_text'].iloc[i][:100]}...")

df[['company_name', 'clean_text']].head(5)

Missing values in clean_text: 0

3 Round Stones, Inc.
Raw  : 3 Round Stones produces a platform for publishing data on the Web. 3 Round Stones provides commercia...
Clean: round stones produces platform publishing data web round stones provides commercial support callimac...

48 Factoring Inc.
Raw  : The company mission is to provide finance to small business. We also provide financing to small busi...
Clean: company mission provide finance small business also provide financing small business bad credit basi...

5PSolutions
Raw  : At 5PSolutions, we wish to make all basic information of different categories easily available to vi...
Clean: psolutions wish make basic information different categories easily available via tablets phones data...


,company_name,clean_text
0,"3 Round Stones, Inc.",round stones produces platform publishing data...
1,48 Factoring Inc.,company mission provide finance small business...
2,5PSolutions,psolutions wish make basic information differe...
3,Abt Associates,abt associates mission driven international co...
4,Accela,accela powers thousands services millions tran...


- 0 missing values left in clean_text, verified on 3 Round Stones, 48 Factoring Inc., and 5PSolutions
- note: I initially used only the description column, but too many companies shared generic words like platform, services, and solutions
- adding company_category and description_short gave stronger domain keywords for each company

## PART 2 - Text Vectorization

### Task 3: Vectorization using TF-IDF

run TfidfVectorizer on clean_text to convert words into numerical weights
- params: max_features=5000 (drops rare noise), ngram_range=(1, 2) (keeps 1-word and 2-word phrases), stop_words='english'
- check matrix shape, sparsity, and sample features


In [6]:
# Fit TfidfVectorizer with 5000 max features and (1, 2) n-grams on clean_text.
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['clean_text'])
sparsity = (1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Non-zero entries: {tfidf_matrix.nnz} | Sparsity: {sparsity:.2f}%")
features = tfidf.get_feature_names_out()
print(f"First 15 features: {list(features[:15])}")
print(f"Middle 15 features: {list(features[2500:2515])}")

TF-IDF matrix shape: (529, 5000)
Non-zero entries: 25731 | Sparsity: 99.03%
First 15 features: ['ability', 'ability monitor', 'able', 'abt associates', 'abuse', 'academic', 'academic markets', 'academics', 'accela', 'accelerate', 'accelrys', 'accelrys provider', 'accenture', 'access', 'access data']
Middle 15 features: ['open data', 'open government', 'open innovation', 'open marketplace', 'open source', 'opencounter', 'opengov', 'opengov com', 'operate', 'operated', 'operates', 'operating', 'operation', 'operational', 'operations']


- TF-IDF matrix shape: (529, 5000) with 25,731 non-zero values (99.03% sparse)
- bigrams kept useful phrases together like open data, open government, and open source

### Task 4: Similarity Computation
- compute pairwise cosine_similarity across all 529 company TF-IDF vectors
- why cosine over Euclidean: cosine checks angle between vectors not length, so short and long descriptions on the same topic still get high similarity
- score stays between 0 and 1

In [7]:
# Compute pairwise cosine similarity across all company TF-IDF vectors.
similarity_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"Similarity matrix shape: {similarity_matrix.shape}")
print(f"Diagonal check: {similarity_matrix[0, 0]:.4f}, {similarity_matrix[1, 1]:.4f}\n")
for j in range(1, 6):
    print(f"{df['company_name'].iloc[0]} vs {df['company_name'].iloc[j]}: {similarity_matrix[0, j]:.4f}")

Similarity matrix shape: (529, 529)
Diagonal check: 1.0000, 1.0000

3 Round Stones, Inc. vs 48 Factoring Inc.: 0.0000
3 Round Stones, Inc. vs 5PSolutions: 0.0273
3 Round Stones, Inc. vs Abt Associates: 0.0000
3 Round Stones, Inc. vs Accela: 0.0249
3 Round Stones, Inc. vs Accenture: 0.0071


## PART 3 - Recommendation Logic

### Task 5: Build Recommendation Function

- write recommend(item_name, top_n=5)
- find company index (case-insensitive with partial match fallback), sort similarity row high to low, skip self-match, return top 5
- test on 3 companies: AccuWeather, Accenture, 48 Factoring Inc.


In [8]:
# Look up a company index, sort its cosine similarity scores, and return top N peers.
def recommend(item_name, top_n=5):
    q = item_name.strip().lower()
    matches = df[df['company_name'].astype(str).str.lower() == q]
    if matches.empty:
        matches = df[df['company_name'].astype(str).str.lower().str.contains(q, na=False)]
    if matches.empty:
        return pd.DataFrame()
    idx = matches.index[0]
    scores = [s for s in sorted(enumerate(similarity_matrix[idx]), key=lambda x: x[1], reverse=True) if s[0] != idx][:top_n]
    rows = []
    for rank, (i, score) in enumerate(scores, 1):
        desc = str(df['description_short'].iloc[i] or df['description'].iloc[i])[:110]
        rows.append({'Rank': rank, 'Company': df['company_name'].iloc[i], 'Category': df['company_category'].iloc[i], 'Similarity': round(float(score), 4), 'Description': desc})
    return pd.DataFrame(rows)
    
print("Recommendation function ready.")

Recommendation function ready.


In [9]:
# Test recommendations for AccuWeather.
print("Recommendations for AccuWeather:")
print(recommend('AccuWeather', top_n=5).to_string(index=False))

Recommendations for AccuWeather:
 Rank                  Company            Category  Similarity                                                                                                    Description
    1         United Mayflower      Transportation      0.1531              United Mayflower Container Services provides local and long-distance moving and storage services.
    2     State Farm Insurance           Insurance      0.1445                      State Farm is a group of insurance and financial services companies in the United States.
    3 Construction Monitor LLC Housing/Real Estate      0.1444 Construction Monitor provides timely, detailed and accurate building permit information, using state of the ar
    4          Weather Channel                          0.1212 The Weather Channel and weather.com provide a national and local weather forecast for cities, as well as weath
    5             SpotHero.com      Transportation      0.1109 SpotHero provides parking at over 

In [10]:
# Test recommendations for Accenture.
print("Recommendations for Accenture:")
print(recommend('Accenture', top_n=5).to_string(index=False))

Recommendations for Accenture:
 Rank                Company                  Category  Similarity                                                                                                    Description
    1    Booz Allen Hamilton     Research & Consulting      0.1510                      Booz Allen Hamilton provides management consulting, technology, and engineering services.
    2           Oliver Wyman     Research & Consulting      0.1282 Oliver Wyman is a global management consulting firm. We combine industry knowledge with expertise in strategy,
    3          R R Donnelley Business & Legal Services      0.0992 RR Donnelley, a global integrated communications provider, uses digital, conventional, and web-based technolog
    4               Deloitte     Research & Consulting      0.0989 Deloitte works in four areas — audit, financial advisory, tax and consulting — and combines the talents of tho
    5 Energy Solutions Forum                    Energy      0.0967 Energy Solut

In [11]:
# Test recommendations for 48 Factoring Inc.
print("Recommendations for 48 Factoring Inc.:")
print(recommend('48 Factoring Inc.', top_n=5).to_string(index=False))

Recommendations for 48 Factoring Inc.:
 Rank           Company             Category  Similarity                                                                                                    Description
    1        DemystData Finance & Investment      0.2538 DemystData helps financial institutions tap into "Big Data" to make better credit decisions. The company’s pro
    2      Credit Karma Finance & Investment      0.2334 Credit Karma is a web-based credit and financial management service that helps more than 20 million consumers 
    3            OnDeck Finance & Investment      0.2196 OnDeck is a technology-powered Main Street lender that uses big data to deliver financing to small businesses.
    4     Credit Sesame Finance & Investment      0.2046 Credit Sesame is a free tool that provides a free credit score, credit monitoring, and a way to save money on 
    5 J.P. Morgan Chase Finance & Investment      0.1776 J.P. Morgan Chase offers investment banking, financial services 

Observations from testing:
- 48 Factoring Inc. -> DemystData (0.2538), Credit Karma (0.2334), OnDeck (0.2196), all finance/credit companies
- Accenture -> Booz Allen Hamilton (0.1510), Oliver Wyman (0.1282), Deloitte (0.0989), all consulting peers
- AccuWeather -> Weather Channel at #4 (0.1212), while United Mayflower got #1 (0.1531) due to shared words like local and national services
- scores around 0.10 to 0.25 are normal for sparse 5000-column TF-IDF vectors

## PART 4 - Simple App Interface

### Task 6: Build UI using Streamlit

- app.py is one readable file that repeats this notebook's steps in order: STEP 1 load -> STEP 2 clean -> STEP 3 TF-IDF -> STEP 4 cosine -> STEP 5 recommend() -> STEP 6 UI
- UI kept simple: company dropdown (type to search), top-N slider (1-10), small info card for the selected company, and a Get recommendations button
- each result card shows the similarity score with a plain-English label (strong >= 0.20, good >= 0.10, weak below), a progress bar, a short description, and the **shared keywords** that caused the match
- a "How does this work?" expander explains TF-IDF and cosine similarity in plain words, plus a table view with CSV download

In [12]:
# Sanity-check app.py (can't import it here because importing it starts a Streamlit page)
import ast

app_src = open('app.py', encoding='utf-8').read()
tree = ast.parse(app_src)   # raises SyntaxError if the file is broken
funcs = [n.name for n in tree.body if isinstance(n, ast.FunctionDef)]
steps = [l.strip() for l in app_src.splitlines() if l.strip().startswith('# STEP')]

print(f"app.py: {len(app_src.splitlines())} lines")
print("functions:", funcs)
print("pipeline sections:")
for s in steps:
    print("  ", s)

app.py: 253 lines
functions: ['get_stopwords', 'clean_text', 'build_model', 'recommend', 'shared_keywords', 'short_desc', 'label_for']
pipeline sections:
   # STEP 2 - text cleaning
   # STEP 1 to 4 - build the model once, then Streamlit caches it.
   # STEP 1: load data
   # STEP 2: handle missing values, combine text columns, clean
   # STEP 3: TF-IDF vectors (one row per company, one column per word/phrase)
   # STEP 4: cosine similarity between every pair of companies
   # STEP 5 - the recommend() function
   # STEP 6 - Streamlit UI


In [13]:
# The app's "Why" line: which words actually produce the similarity score?
# TF-IDF rows are already L2-normalised, so cosine(a, b) = sum over words of w_a * w_b.
# The biggest products are therefore the biggest reasons for a match.
def shared_keywords(a_name, b_name, k=5):
    a = df.index[df['company_name'] == a_name][0]
    b = df.index[df['company_name'] == b_name][0]
    prod = tfidf_matrix[a].multiply(tfidf_matrix[b]).toarray().ravel()
    top = [features[j] for j in prod.argsort()[::-1][:15] if prod[j] > 0]
    phrases = [t for t in top if ' ' in t]                      # keep "united states", drop "united"
    words = [t for t in top if not any(t in p.split() for p in phrases)]
    return words[:k], prod.sum(), similarity_matrix[a, b]

for a, b in [('Accenture', 'Booz Allen Hamilton'), ('48 Factoring Inc.', 'OnDeck'), ('AccuWeather', 'United Mayflower')]:
    kw, total, cos = shared_keywords(a, b)
    print(f"{a} <-> {b}: sum of products = {total:.4f}, cosine = {cos:.4f}")
    print(f"   shared keywords: {kw}")

Accenture <-> Booz Allen Hamilton: sum of products = 0.1510, cosine = 0.1510
   shared keywords: ['groups', 'industry', 'provides management', 'management consulting', 'services']
48 Factoring Inc. <-> OnDeck: sum of products = 0.2196, cosine = 0.2196
   shared keywords: ['credit', 'small business', 'financing', 'financial', 'company']
AccuWeather <-> United Mayflower: sum of products = 0.1531, cosine = 0.1531
   shared keywords: ['provides local', 'united states', 'services']


- sum of word-by-word products matches the cosine value exactly, so the keyword list is a real explanation of the score, not a guess
- this also explains the odd AccuWeather -> United Mayflower result from Task 5: the match comes from generic phrases like *united states* and *local*, not from weather. TF-IDF only sees shared words, not meaning (Word2Vec from Assignment 19 would handle that better)
- note: switched from `@st.cache_data` to `@st.cache_resource` for the model build, since the vectorizer and sparse matrix don't need to be copied on every rerun
- note: the app tries NLTK stopwords first and falls back to scikit-learn's built-in list if the download fails, so the server can't crash on a network hiccup
- dropped the old src/app.py copy, one app.py is easier to keep in sync

## PART 5 - Version Control with Git & GitHub

### Task 7: Git & GitHub Setup

GitHub repository: https://github.com/INEEDTHATGT3/content-based-recommender

- repo contains app.py, requirements.txt, README.md, this notebook, data/us_companies.csv, src/recommender.py, tests/, render.yaml and .python-version
- versions in requirements.txt are pinned to the exact ones I tested with, so Render installs the same thing every time
- had to rename Readme.md -> README.md: Windows ignores filename case but GitHub/Linux doesn't, so GitHub wasn't showing it on the repo page

In [14]:
# Write requirements.txt (pinned versions = same environment locally and on Render)
requirements_content = """streamlit==1.66.1
pandas==3.0.5
numpy==2.5.3
scikit-learn==1.9.1
nltk==3.10.4
"""

with open('requirements.txt', 'w', encoding='utf-8') as f:
    f.write(requirements_content)
print(open('requirements.txt').read())

streamlit==1.66.1
pandas==3.0.5
numpy==2.5.3
scikit-learn==1.9.1
nltk==3.10.4



Git commands used (run in a terminal from the project folder):

```bash
git init -b main
git add .
git commit -m "Content-based company recommender: notebook, Streamlit app, Render config"
git remote add origin https://github.com/INEEDTHATGT3/content-based-recommender.git
git push -u origin main
```

## PART 6 - Deployment on Render

### Task 8: Deploy Application on Render

- Render dashboard -> New -> Web Service -> pasted the GitHub repo URL -> Render auto-detected Python

| Setting | Value |
|---|---|
| Runtime | Python 3 (version from `.python-version` = 3.13) |
| Branch | main |
| Region | Singapore (closest to India) |
| Instance type | Free |
| Build command | `pip install -r requirements.txt` |
| Start command | `streamlit run app.py --server.port $PORT --server.address 0.0.0.0` |

- note: `--server.port $PORT` and `0.0.0.0` are required because Render gives the app a dynamic port and the health check fails if Streamlit stays on its default local port 8501
- the same settings are saved in `render.yaml`, so New -> Blueprint on Render recreates the service in one click
- free instances sleep after ~15 minutes of no traffic, so the first visit after that takes about a minute to wake up

### Task 9: Final Validation

**Deployed Application URL: https://content-based-recommender-7aei.onrender.com**

- opened the live link: dropdown loads all 529 companies, no file path errors
- AccuWeather, Accenture and 48 Factoring Inc. give the same top matches as the notebook (Booz Allen Hamilton / Oliver Wyman / Deloitte for Accenture, DemystData / Credit Karma / OnDeck for 48 Factoring) and never recommend the company itself
- learnings: content-based filtering works right away on new company text without user history (no cold-start problem), and combining 3 text columns worked much better than description alone

Summary and Personal Learnings:
1. Content-Based vs. Collaborative Filtering: Content-based filtering does not suffer from the user cold-start problem because it only relies on item metadata and descriptions. As soon as a new company is added, its description can be vectorized immediately.
2. Feature Engineering: Combining multiple text attributes (description, category, short_description) provides much better discrimination than raw descriptions alone.
3. Explainability: because TF-IDF cosine is just a sum of word-by-word products, every recommendation can be explained by its shared keywords - that made debugging the AccuWeather result easy.
4. Deployment: caching the model build (`@st.cache_resource`), pinning versions and binding to Render's `$PORT` were the three things that made the deploy work first time.